# 03d — Two-Stage Grid · REG (Stage 2)

**역할**: 회귀기 1개를 **y>0 die만** 학습 → die-level reg_pred csv 저장. 03e에서 03c clf prob과 곱.

- 전처리 = `final/modules/preprocess.run` DEFAULT_PARAMS (= baseline 1차 best)
- 학습 = die-level conditional regression (`y_positive_only=True`)
- target = `log1p(y)` (재변환 후 unit RMSE 평가)
- objective = OOF unit RMSE — clf 와는 독립 학습 (`add_clf_proba_to_reg=False` 기본)
- `REG_MODEL_NAME` 스위치 → 모델 1개 = 1번 실행. **코랩 병렬**
- 손실함수 자동 탐색 — lgbm: `regression/poisson/tweedie_1.2/tweedie_1.5`, xgb: `squarederror/tweedie_*`, catboost: `RMSE/Tweedie_*`

출력: `4_output/final/two_stage_grid/reg/{REG_MODEL_NAME}/` (oof/val/test die+unit + fold_models + best_params)

## 1. 환경 설정 + 모듈 import

In [1]:
import os, sys

GDRIVE_FINAL_ID = '1HR7LlQmp4n9wGh2WneyVex2mCZ-poiY9'   # ★ Colab에서 final.zip 업로드 후 공유 ID 입력

try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system('gdown 1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system('gdown 1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system('gdown 1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    if not os.path.exists('/content/project/3_modeling/final/modules/hpo.py'):
        assert GDRIVE_FINAL_ID, 'GDRIVE_FINAL_ID가 비어있음 — final.zip 공유 ID를 입력하세요'
        os.makedirs('/content/project/3_modeling/final', exist_ok=True)
        os.system(f'gdown {GDRIVE_FINAL_ID} -O /content/final.zip')
        os.system('unzip -qo /content/final.zip -d /content/project/3_modeling/final')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs
from utils.evaluate import rmse

MODEL_ROOT = os.path.join(PROJECT_ROOT, '3_modeling')
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)

from final.modules import preprocess, hpo, models

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'Available REG models: {models.AVAILABLE_MODELS}')

setup 완료
PROJECT_ROOT = c:\Users\Dell5371\Desktop\기업연계프로젝트
Available REG models: ['lgbm', 'xgb', 'catboost', 'et', 'enet', 'zitboost']


## 2. 실험 설정

- `REG_MODEL_NAME` 스위치: 한 번 실행 = 한 모델 학습.
- `TARGET_TRANSFORM='log1p'` 고정 (Two-Stage Stage 2 정석 + zero-skew 분포 대응)
- `Y_POSITIVE_ONLY=True` 고정 — **y>0 die 만 학습** (conditional regression `E[Y|Y>0,x]`)
- `PARAMS = {}` 빈 dict → `final/modules/preprocess.py` DEFAULT 사용 (= baseline 1차 best)

In [ ]:
# ── 모델 선택 (한 번에 1개) ──
REG_MODEL_NAME = 'lgbm'        # ★ 'lgbm' | 'xgb' | 'catboost' | 'et' | 'enet'
assert REG_MODEL_NAME in {'lgbm', 'xgb', 'catboost', 'et', 'enet'}, \
    f'REG_MODEL_NAME invalid: {REG_MODEL_NAME}'

# ── 실험 식별 ──
EXP_ID   = f'ts-reg-{REG_MODEL_NAME}-001'
EXP_MEMO = f'Two-Stage Grid · REG · {REG_MODEL_NAME} · y>0 conditional · log1p'
USER     = 'jh'

# ── Optuna 예산 ──
N_TRIALS = 1
N_FOLDS  = 5

# ── Two-Stage Stage 2 정책 (변경 금지 — 곱셈 구조 일관성) ──
TARGET_TRANSFORM = 'log1p'
Y_POSITIVE_ONLY  = True
CLIP_Y_EXTREME   = True

# ── 출력 경로 ──
OUT_DIR = os.path.join(OUTPUT_DIR, 'final', 'two_stage_grid', 'reg', REG_MODEL_NAME)
DB_PATH = os.path.join(OUT_DIR, f'optuna_{USER}_{EXP_ID}.db')
os.makedirs(OUT_DIR, exist_ok=True)

# ── 전처리 PARAMS — 빈 dict → preprocess.py DEFAULT (baseline 1차 best) ──
PARAMS = {}

# ── 디바이스 ──
# models.DEVICE = 'gpu'

print(f'EXP: {EXP_ID} | USER: {USER}')
print(f'REG_MODEL_NAME: {REG_MODEL_NAME}')
print(f'N_TRIALS={N_TRIALS}, N_FOLDS={N_FOLDS}')
print(f'TARGET_TRANSFORM={TARGET_TRANSFORM} | Y_POSITIVE_ONLY={Y_POSITIVE_ONLY} | CLIP_Y_EXTREME={CLIP_Y_EXTREME}')
print(f'OUT_DIR={OUT_DIR}')
print(f'DEVICE={models.DEVICE}')
print(f'PARAMS override: {PARAMS}  (빈 dict → preprocess.py DEFAULT_PARAMS)')

## 3. 데이터 로드 + Y clip + transform 함수 + 전처리

In [3]:
# ── 데이터 로드 ──
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)
print(f'xs: {xs.shape}, feat_cols: {len(feat_cols)}')

# ── y_train 극단값 clip ──
ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개')

# ── target transform = log1p (고정) ──
target_transform_fn = lambda y: np.log1p(np.asarray(y))
target_inverse_fn   = lambda y: np.clip(np.expm1(np.asarray(y)), 0.0, None)
print(f'[target transform] log1p / expm1(clip>=0)')

# ── 전처리 ──
pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PARAMS)
xs_train = pp['xs_train']
xs_val   = pp['xs_val']
xs_test  = pp['xs_test']
feat_cols_clean = pp['feat_cols']
print(f'\n[전처리 완료] feat_cols: {len(feat_cols_clean)}')
print(f'  xs_train: {xs_train.shape}, val: {xs_val.shape}, test: {xs_test.shape}')

# ── y>0 분포 확인 ──
yt = ys_input['train'][TARGET_COL]
print(f'\nUnit y>0 비율 (train): {(yt > 0).mean():.4f} ({(yt > 0).sum():,} unit)')
print(f'E[Y | Y>0]            = {yt[yt > 0].mean():.6f}')

[load_xs] all-NaN 행 407개 제거 → 174,573행
[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572
[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
xs: (174572, 1091), feat_cols: 1087
[CLIP_Y_EXTREME] 1.0 → 0.097417 clip, 1개
[target transform] log1p / expm1(clip>=0)
[Stage 0] 웨이퍼맵 사전 제외: 1087 → 1033 (54개 제거)
클리닝 파이프라인 시작
원본 feature 수: 1033
[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 928개
    컬럼: 1033 → 928 (105개 제거)
    DataFrame: (104748, 986)

[고결측 제거] threshold=40%
  제거: 5개, 잔여: 923개
    컬럼: 928 → 923 (5개 제거)
    DataFrame: (104748, 981)

[중복 컬럼 제거] sample_n=5000
  제거: 27개, 잔여: 896개
    컬럼: 923 → 896 (27개 제거)
    DataFrame: (104748, 954)

[고상관 제거] threshold=0.9, keep_by=std (std)
  제거: 332개, 잔여: 564개
    컬럼: 896 → 564 (332개 제거)
    DataFrame: (104748, 622)

[결측 indicator] 9개 컬럼 추가 (결측률 >= 5%)
[공간 보간 imputation] 

## 4. Optuna HPO (REG, y>0 die만 학습)

- `run_hpo(y_positive_only=True)`: fold마다 y_die==0 die 제외 후 학습
- target = `log1p(y_die)` (broadcast된 unit y), 예측 후 `expm1`+clip
- objective = OOF die-level pred → unit mean → unit RMSE (clf 무관, conditional reg 평가)
- 손실함수 (lgbm/xgb/catboost) 는 search space 의 categorical 로 자동 탐색

In [4]:
study_meta = {
    'exp_id':           EXP_ID,
    'exp_memo':         EXP_MEMO,
    'user':             USER,
    'reg_model_name':   REG_MODEL_NAME,
    'target_transform': TARGET_TRANSFORM,
    'y_positive_only':  Y_POSITIVE_ONLY,
    'clip_y_extreme':   CLIP_Y_EXTREME,
    'effective_pp_params': pp['effective_params'],
    'n_trials':         N_TRIALS,
    'n_folds':          N_FOLDS,
    'seed':             SEED,
}

res = hpo.run_hpo(
    xs_train=xs_train,
    ys_train_unit=ys_input['train'],
    feat_cols=feat_cols_clean,
    model_name=REG_MODEL_NAME,
    n_trials=N_TRIALS,
    n_folds=N_FOLDS,
    y_positive_only=Y_POSITIVE_ONLY,
    target_transform_fn=target_transform_fn,
    target_inverse_fn=target_inverse_fn,
    study_name=EXP_ID,
    storage=f'sqlite:///{DB_PATH}',
    user_attrs=study_meta,
    # ★ trial별 holdout RMSE 기록
    xs_val=xs_val,   ys_val_unit=ys_input['validation'],
    xs_test=xs_test, ys_test_unit=ys_input['test'],
)
study       = res['study']
best_params = res['best_params']

print(f'\n[HPO 완료] best OOF RMSE = {res["best_value"]:.6f}')
print(f'best_params = {best_params}')

[I 2026-05-02 16:26:02,956] A new study created in RDB with name: ts-reg-lgbm-001


  0%|          | 0/1 [00:00<?, ?it/s]

[I 2026-05-02 16:26:14,005] Trial 0 finished with value: 0.008197561631265159 and parameters: {'n_estimators': 1186, 'learning_rate': 0.24517932047070642, 'num_leaves': 283, 'max_depth': 10, 'min_child_samples': 66, 'subsample': 0.5779972601681014, 'colsample_bytree': 0.15227525095137953, 'reg_alpha': 1.6175449623854197, 'reg_lambda': 0.002570603566117598, 'min_split_gain': 0.0023585940584142655, 'path_smooth': 1.0292247147901223, 'objective': 'regression'}. Best is trial 0 with value: 0.008197561631265159.

[HPO 완료] best OOF RMSE = 0.008198
best_params = {'n_estimators': 1186, 'learning_rate': 0.24517932047070642, 'num_leaves': 283, 'max_depth': 10, 'min_child_samples': 66, 'subsample': 0.5779972601681014, 'colsample_bytree': 0.15227525095137953, 'reg_alpha': 1.6175449623854197, 'reg_lambda': 0.002570603566117598, 'min_split_gain': 0.0023585940584142655, 'path_smooth': 1.0292247147901223, 'objective': 'regression'}


## 5. Best trial 재학습 (K-fold OOF) + die-level reg_pred 캡쳐

In [5]:
final = hpo.refit_best(
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    ys_train_unit=ys_input['train'],
    feat_cols=feat_cols_clean,
    model_name=REG_MODEL_NAME,
    best_params=best_params,
    n_folds=N_FOLDS,
    y_positive_only=Y_POSITIVE_ONLY,
    target_transform_fn=target_transform_fn,
    target_inverse_fn=target_inverse_fn,
)

# ── unit RMSE 계산 (reg 단독, conditional reg 평가) ──
y_train_true = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
y_val_true   = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_test_true  = ys_input['test'].set_index(KEY_COL)[TARGET_COL]

oof_unit  = final['oof_pred_unit'].set_index(KEY_COL)['pred'].loc[y_train_true.index]
val_unit  = final['val_pred_unit'].set_index(KEY_COL)['pred'].loc[y_val_true.index]
test_unit = final['test_pred_unit'].set_index(KEY_COL)['pred'].loc[y_test_true.index]

oof_rmse  = float(np.sqrt(np.mean((oof_unit.values  - y_train_true.values) ** 2)))
val_rmse  = float(np.sqrt(np.mean((val_unit.values  - y_val_true.values)   ** 2)))
test_rmse = float(np.sqrt(np.mean((test_unit.values - y_test_true.values)  ** 2)))

print(f'\n[Refit 완료] (reg 단독, y>0 conditional, log1p)')
print(f'  OOF  unit RMSE = {oof_rmse:.6f}')
print(f'  val  unit RMSE = {val_rmse:.6f}')
print(f'  test unit RMSE = {test_rmse:.6f}')
print(f'fold_models: {len(final["fold_models"])}개')
print(f'  → 단독 평가: clf와 곱셈 안 한 conditional reg pred 의 unit RMSE')
print(f'  → 03e 에서 clf prob과 곱셈한 후가 진짜 final RMSE')

[refit fold 1/5] tr_units=20949, vl_units=5238
[refit fold 2/5] tr_units=20949, vl_units=5238
[refit fold 3/5] tr_units=20950, vl_units=5237
[refit fold 4/5] tr_units=20950, vl_units=5237
[refit fold 5/5] tr_units=20950, vl_units=5237

[Refit 완료] (reg 단독, y>0 conditional, log1p)
  OOF  unit RMSE = 0.008198
  val  unit RMSE = 0.008330
  test unit RMSE = 0.010322
fold_models: 5개
  → 단독 평가: clf와 곱셈 안 한 conditional reg pred 의 unit RMSE
  → 03e 에서 clf prob과 곱셈한 후가 진짜 final RMSE


## 6. 산출물 저장 (`4_output/final/two_stage_grid/reg/{MODEL_NAME}/`)

기존 `hpo.save_artifacts` 그대로 활용 (postprocess는 03e 단계에서 수행하므로 여기선 단순 저장만).

In [6]:
hpo.save_artifacts(
    refit_result=final,
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    out_dir=OUT_DIR, exp_id=EXP_ID,
    feature_names=feat_cols_clean,
    extra_feature_name=None,
    y_train_unit=ys_input['train'],
    y_val_unit=ys_input['validation'],
    y_test_unit=ys_input['test'],
    postprocess_config=None,   # ★ 03e 에서 곱셈 후 후처리. 여기선 단순 mean 집계.
    study_meta=study_meta,
)

for f_ in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, f_)) / 1024
    print(f'  {f_:30s}  {sz:>10,.1f} KB')

# ── Colab → 로컬 자동 다운로드 (로컬은 자동 skip) ──
try:
    import google.colab
    from google.colab import files
    import shutil
    _zip_base = os.path.join('/content', f'reg_{REG_MODEL_NAME}_{EXP_ID}_outputs')
    _zip_path = shutil.make_archive(_zip_base, 'zip', OUT_DIR)
    print(f'[zip 생성] {_zip_path} ({os.path.getsize(_zip_path)/1024:.1f} KB)')
    try:
        files.download(_zip_path)
        print(f'[브라우저 다운로드 트리거] {os.path.basename(_zip_path)}')
    except Exception as _e:
        from IPython.display import FileLink, display
        print(f'[files.download 실패: {_e}] 아래 링크 클릭해 수동 다운로드')
        display(FileLink(_zip_path))
except ImportError:
    pass

[save_artifacts] c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\_temp\two_stage_grid\reg\lgbm 저장 완료 (fold_models.pkl + best_params.json + 6 CSV, unit=mean)
  best_params.json                       9.1 KB
  fold_models.pkl                      105.8 KB
  oof_die.csv                        5,503.2 KB
  oof_unit.csv                         950.0 KB
  optuna_jh_ts-reg-lgbm-001.db         112.0 KB
  test_die.csv                       1,840.9 KB
  test_unit.csv                        318.3 KB
  val_die.csv                        1,841.2 KB
  val_unit.csv                         318.3 KB


## 7. 요약

In [7]:
print('=' * 75)
print(f'  REG · {REG_MODEL_NAME} ({EXP_ID})')
print('=' * 75)
print(f'  feat cols (clean) : {len(feat_cols_clean)}')
print(f'  trials            : {len(study.trials)}')
print(f'  transform         : {TARGET_TRANSFORM}')
print(f'  y_positive_only   : {Y_POSITIVE_ONLY}  (y>0 die만 학습)')
print('-' * 75)
print(f'  {"":12s}  {"OOF":>11s}  {"val":>11s}  {"test":>11s}')
print(f'  {"unit RMSE":12s}  {oof_rmse:11.6f}  {val_rmse:11.6f}  {test_rmse:11.6f}')
print('-' * 75)
print(f'  → die-level reg csv 저장 완료. 03e_ts_combine.ipynb 에서 clf prob과 곱.')
print(f'  → 다른 REG 모델 돌리려면 REG_MODEL_NAME 바꿔서 재실행.')
print('=' * 75)

  REG · lgbm (ts-reg-lgbm-001)
  feat cols (clean) : 573
  trials            : 1
  transform         : log1p
  y_positive_only   : True  (y>0 die만 학습)
---------------------------------------------------------------------------
                        OOF          val         test
  unit RMSE        0.008198     0.008330     0.010322
---------------------------------------------------------------------------
  → die-level reg csv 저장 완료. 03e_ts_combine.ipynb 에서 clf prob과 곱.
  → 다른 REG 모델 돌리려면 REG_MODEL_NAME 바꿔서 재실행.
